# Tree models (Decision Tree, Random Forest)

Target: `final_grade_pct` (regression). This notebook loads the prepared train/test data saved by `Project.ipynb`, so run that notebook first. It also needs `results/linear.json` from `Project-Linear.ipynb` for the combined comparison at the end.

Every model is evaluated on the same 10 outer folds of the training set, using the predictors kept by the Lasso selection step. Each model is fit on the outer training folds, and its validation RMSE is recorded. It is then refitted on all training rows and scored once on the test set.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path('prepared_data')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / 'meta.json') as f:
    meta = json.load(f)

SEED = meta['seed']
N_SPLITS = meta['n_splits']
PREDICTOR_COLS = meta['predictor_cols']
NUMERIC_COLS = meta['numeric_cols']

X_train_final = pd.read_csv(DATA_DIR / 'X_train.csv', index_col=0)
X_test_final = pd.read_csv(DATA_DIR / 'X_test.csv', index_col=0)
Y_train = pd.read_csv(DATA_DIR / 'Y_train.csv', index_col=0)[meta['target']]
Y_test = pd.read_csv(DATA_DIR / 'Y_test.csv', index_col=0)[meta['target']]
print('train:', X_train_final.shape, ' test:', X_test_final.shape)

## Shared setup

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

MODEL_COLS = list(X_train_final.columns)
ALPHA_GRID = np.logspace(-4, 1, 60)
RIDGE_GRID = np.logspace(-3, 3, 50)
inner_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
outer_folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(X_train_final))


def source_variable(col):
    # Maps a one-hot column back to its raw predictor name
    for raw in PREDICTOR_COLS:
        if col == raw or col.startswith(raw + '_'):
            return raw
    return col


def run_group(factories):
    # factories maps a model name to a function that returns a fresh, unfitted model
    cv_fold_rmse = {name: [] for name in factories}
    for fold, (train_idx, valid_idx) in enumerate(outer_folds, start=1):
        X_tr, X_valid = X_train_final.iloc[train_idx], X_train_final.iloc[valid_idx]
        Y_tr, Y_valid = Y_train.iloc[train_idx], Y_train.iloc[valid_idx]
        for name, make in factories.items():
            model = make().fit(X_tr, Y_tr)
            cv_fold_rmse[name].append(root_mean_squared_error(Y_valid, model.predict(X_valid)))
        print(f'Finished outer fold {fold}/{len(outer_folds)}')

    fitted, rows = {}, []
    for name, make in factories.items():
        model = make().fit(X_train_final, Y_train)
        if isinstance(model, GridSearchCV):
            print(f'{name} selected on all training rows: {model.best_params_}')
            fitted[name] = model.best_estimator_
        else:
            fitted[name] = model
        rows.append({
            'model': name,
            'cv_rmse': np.mean(cv_fold_rmse[name]),
            'cv_rmse_sd': np.std(cv_fold_rmse[name], ddof=1),
            'test_rmse': root_mean_squared_error(Y_test, model.predict(X_test_final)),
        })
    table = pd.DataFrame(rows).set_index('model')
    print(table.round(3).to_string())
    return {'cv_fold_rmse': cv_fold_rmse, 'fitted': fitted, 'table': table}


def scaled_model(cols, estimator):
    scale_cols = [c for c in cols if source_variable(c) in NUMERIC_COLS]
    scaling = ColumnTransformer([('scale', StandardScaler(), scale_cols)],
                                remainder='passthrough', verbose_feature_names_out=False)
    return make_pipeline(scaling.set_output(transform='pandas'), estimator)

### Tree models (Decision Tree, Random Forest)

Trees are invariant to scaling, so they use the raw predictors. Each model tunes its complexity with the same inner 10-fold CV.

In [ ]:
def tune_tree():
    return GridSearchCV(DecisionTreeRegressor(random_state=SEED),
                        {'max_depth': [3, 4, 5, 6, 8, 10], 'min_samples_leaf': [10, 20, 50]},
                        cv=inner_cv, scoring='neg_root_mean_squared_error', n_jobs=-1)


def tune_forest():
    return GridSearchCV(RandomForestRegressor(n_estimators=200, random_state=SEED),
                        {'max_features': [0.33, 0.6, 1.0], 'min_samples_leaf': [5, 20]},
                        cv=inner_cv, scoring='neg_root_mean_squared_error', n_jobs=-1)


tree_results = run_group({
    'Decision Tree': tune_tree,
    'Random Forest': tune_forest,
})

importance = pd.Series(tree_results['fitted']['Random Forest'].feature_importances_, index=MODEL_COLS).sort_values()
importance.plot(kind='barh', figsize=(8, 5))
plt.title('Random Forest feature importance (impurity-based, all training rows)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

### Combined comparison

All five models plus the mean-only baseline side by side, with paired tests on outer-fold errors. The linear models and baseline come from `Project-Linear.ipynb`, which uses the same data and the same outer folds. These tests are exploratory because folds share training observations.

In [ ]:
linear_path = RESULTS_DIR / 'linear.json'
if not linear_path.exists():
    raise FileNotFoundError('Run Project-Linear.ipynb first to create results/linear.json')
with open(linear_path) as f:
    linear_saved = json.load(f)

table_all = pd.concat([pd.DataFrame.from_dict(linear_saved['table'], orient='index'), tree_results['table']])
cv_fold_rmse = {**linear_saved['cv_fold_rmse'], **tree_results['cv_fold_rmse']}
print(table_all.round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot([cv_fold_rmse[name] for name in table_all.index], showmeans=True)
ax.set_xticks(range(1, len(table_all) + 1), table_all.index, rotation=10)
ax.set_ylabel('Outer-fold RMSE')
ax.set_title('Paired 10-fold cross-validation errors')
plt.tight_layout()
plt.show()

for left, right in [
    ('Mean only (baseline)', 'OLS'),
    ('OLS', 'Ridge'),
    ('OLS', 'Lasso'),
    ('OLS', 'Decision Tree'),
    ('OLS', 'Random Forest'),
]:
    p_value = ttest_rel(cv_fold_rmse[left], cv_fold_rmse[right]).pvalue
    print(f'Paired t-test, {left} vs {right} (exploratory): p = {p_value:.4g}')

## Conclusion

The linear models are the strongest on this pipeline. Every model clearly beats the mean-only baseline, the linear models are statistically tied, and the tree-based models are worse than OLS.

- **Trees are worse.** Random Forest has CV RMSE 9.78 (max_features 0.33, min_samples_leaf 20), and the Decision Tree has 9.99 (max_depth 6, min_samples_leaf 50). Both are significantly worse than OLS (p ≈ 2e-5 and 6e-7).
- **Test set agrees.** Refitted on all training rows and scored once on the 1,800-row test set: mean only 11.71, OLS/Ridge/Lasso 9.83, Random Forest 9.94, Decision Tree 10.23.

**Interpretation.** Most of the signal is close to linear and additive. The trees split the data into coarse regions and lose accuracy at this sample size.

**Caveats.** The target was kept on its original scale because log(Y) makes its skew worse (−0.51 to −2.55). No outlier capping was applied. Imputation statistics, correlation pruning, and the Lasso fit were computed once on the full training set before the outer CV, so the CV scores are somewhat optimistic. No separate nested check is run. The paired tests treat folds as independent, but they share training rows, so the p-values are exploratory. Results depend on the random seed.